# HydroSense AI - Data Understanding

This notebook validates and describes the 2205-cycle hydraulic test-rig dataset. Raw files are read only.

## 1. Setup
The project paths come from `src.config`, and sensor files are loaded only one at a time.

In [3]:
from pathlib import Path
import sys
import gc

import pandas as pd
import matplotlib.pyplot as plt

# Find the project root whether the notebook runs from /notebooks or from the root
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

# Make the src package importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python used by this kernel:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("pandas version:", pd.__version__)

Python used by this kernel: c:\Users\o3995\anaconda3\python.exe
Project root: C:\Users\o3995\Downloads\ML_SUT_Projects\Individual_Lab_ML_SUT\Hydraulic_Systems\System
pandas version: 2.3.3


In [4]:
from src import config
from src.data_loading import load_profile, load_sensor, validate_dataset
from src.eda import (
    compute_cycle_summary,
    count_outliers_iqr,
    check_label_combinations,
    row_order_check,
    sensor_quality_checks,
)
from src.utils import set_plot_style, save_figure

set_plot_style()

## 2. File inventory
This section lists every file in `dataset/raw` and reports its size. The raw dataset contains 20 files.

In [5]:
inventory = pd.DataFrame([
    {"file": p.name, "size_MiB": round(p.stat().st_size / 1024**2, 2)}
    for p in sorted(config.RAW_DIR.iterdir()) if p.is_file()
])
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
inventory.to_csv(config.RESULTS_DIR / "raw_file_inventory.csv", index=False)
print(f"Files: {len(inventory)}")
print(f"Total MiB: {inventory['size_MiB'].sum():.2f}")
display(inventory)

Files: 20
Total MiB: 530.49


,file,size_MiB
0,CE.txt,0.87
1,CP.txt,0.74
2,description.txt,0.00
3,documentation.txt,0.00
4,EPS1.txt,83.36
5,FS1.txt,7.23
6,FS2.txt,7.89
7,profile.txt,0.03
8,PS1.txt,87.16
9,PS2.txt,78.56


## 3. Sensor and target descriptions

In [6]:
sensor_table = pd.DataFrame([
    {**meta, "sensor": name,
     "what_it_measures": f"Records {meta['group']} in each 60-second cycle."}
    for name, meta in config.SENSORS.items()
])
sensor_table = sensor_table[["sensor", "group", "unit", "sampling_hz", "samples_per_cycle", "what_it_measures"]]
sensor_table.to_csv(config.RESULTS_DIR / "sensor_description_table.csv", index=False)

target_rows = []
for target, labels in config.TARGET_MEANINGS.items():
    for value, meaning in labels.items():
        target_rows.append({"column": target, "value": value, "meaning": meaning,
                            "number_of_classes": len(labels)})
target_table = pd.DataFrame(target_rows)
target_table.to_csv(config.RESULTS_DIR / "target_description_table.csv", index=False)
print("Sensors")
display(sensor_table)
print("Targets")
display(target_table)

Sensors


,sensor,group,unit,sampling_hz,samples_per_cycle,what_it_measures
0,PS1,pressure,bar,100,6000,Records pressure in each 60-second cycle.
1,PS2,pressure,bar,100,6000,Records pressure in each 60-second cycle.
2,PS3,pressure,bar,100,6000,Records pressure in each 60-second cycle.
3,PS4,pressure,bar,100,6000,Records pressure in each 60-second cycle.
4,PS5,pressure,bar,100,6000,Records pressure in each 60-second cycle.
5,PS6,pressure,bar,100,6000,Records pressure in each 60-second cycle.
6,EPS1,motor power,W,100,6000,Records motor power in each 60-second cycle.
7,FS1,flow,l/min,10,600,Records flow in each 60-second cycle.
8,FS2,flow,l/min,10,600,Records flow in each 60-second cycle.
9,TS1,temperature,deg C,1,60,Records temperature in each 60-second cycle.


Targets


,column,value,meaning,number_of_classes
0,cooler,3,near failure,3
1,cooler,20,reduced efficiency,3
2,cooler,100,full efficiency,3
3,valve,100,optimal switching behavior,4
4,valve,90,small lag,4
5,valve,80,severe lag,4
6,valve,73,near failure,4
7,pump,0,no leakage,3
8,pump,1,weak leakage,3
9,pump,2,severe leakage,3


## 4. Dataset validation

In [7]:
validation = validate_dataset()
validation.to_csv(config.RESULTS_DIR / "validation_summary.csv", index=False)
display(validation)

,file,rows,cols,size MiB,NaN count,status
0,profile.txt,2205,5,0.03,0,OK
1,PS1.txt,2205,6000,87.16,0,OK
2,PS2.txt,2205,6000,78.56,0,OK
3,PS3.txt,2205,6000,66.68,0,OK
4,PS4.txt,2205,6000,44.53,0,OK
5,PS5.txt,2205,6000,74.17,0,OK
6,PS6.txt,2205,6000,74.27,0,OK
7,EPS1.txt,2205,6000,83.36,0,OK
8,FS1.txt,2205,600,7.23,0,OK
9,FS2.txt,2205,600,7.89,0,OK


## 5. Condition profile
The profile is shown from both ends, along with its data types and shape.

In [8]:
profile = load_profile()
print("Shape:", profile.shape)
print("Dtypes:")
print(profile.dtypes)
print("First 5 rows:")
display(profile.head())
print("Last 5 rows:")
display(profile.tail())

Shape: (2205, 5)
Dtypes:
cooler         int64
valve          int64
pump           int64
accumulator    int64
stable         int64
dtype: object
First 5 rows:


,cooler,valve,pump,accumulator,stable
0,3,100,0,130,1
1,3,100,0,130,1
2,3,100,0,130,1
3,3,100,0,130,1
4,3,100,0,130,1


Last 5 rows:


,cooler,valve,pump,accumulator,stable
2200,100,100,0,90,0
2201,100,100,0,90,0
2202,100,100,0,90,0
2203,100,100,0,90,0
2204,100,100,0,90,0


## 6. Sensor shapes and small raw slices
Each sensor is loaded separately. Only the first and last five values from cycle 0 and the final cycle are printed.

In [9]:
slice_rows = []
for name in config.SENSORS:
    values = load_sensor(name)
    print(name, "shape=", values.shape, "dtype=", values.dtype)
    print(" cycle 0 first 5:", values[0, :5].tolist())
    print(" cycle 0 last 5:", values[0, -5:].tolist())
    print(" final cycle first 5:", values[-1, :5].tolist())
    print(" final cycle last 5:", values[-1, -5:].tolist())
    slice_rows.append({"sensor": name, "shape": str(values.shape), "dtype": str(values.dtype),
                       "first_cycle_first5": values[0, :5].tolist(),
                       "first_cycle_last5": values[0, -5:].tolist(),
                       "last_cycle_first5": values[-1, :5].tolist(),
                       "last_cycle_last5": values[-1, -5:].tolist()})
    del values
    gc.collect()
slice_table = pd.DataFrame(slice_rows)
slice_table.to_csv(config.RESULTS_DIR / "sensor_slice_checks.csv", index=False)

PS1 shape= (2205, 6000) dtype= float32
 cycle 0 first 5: [151.47000122070312, 151.4499969482422, 151.52000427246094, 151.27000427246094, 150.8000030517578]
 cycle 0 last 5: [151.16000366210938, 151.13999938964844, 151.10000610351562, 151.2100067138672, 151.19000244140625]
 final cycle first 5: [151.8300018310547, 151.77999877929688, 151.77000427246094, 151.64999389648438, 151.49000549316406]
 final cycle last 5: [151.63999938964844, 151.6999969482422, 151.72999572753906, 151.7100067138672, 151.6999969482422]
PS2 shape= (2205, 6000) dtype= float32
 cycle 0 first 5: [125.5, 125.38999938964844, 125.4000015258789, 125.02999877929688, 124.05000305175781]
 cycle 0 last 5: [124.91000366210938, 124.9800033569336, 125.11000061035156, 125.13999938964844, 125.0999984741211]
 final cycle first 5: [125.87999725341797, 125.86000061035156, 125.83999633789062, 125.4800033569336, 125.02999877929688]
 final cycle last 5: [125.7300033569336, 125.80999755859375, 125.7699966430664, 125.66000366210938, 125.

## 7. Cycle alignment
All sensor arrays have 2205 rows. The row at index `i` represents the same operating cycle in every file and in `profile.txt`, because the data has no ID column.

In [10]:
for name in config.SENSORS:
    values = load_sensor(name)
    assert values.shape[0] == config.N_CYCLES
    del values
    gc.collect()
print("All 17 sensor files have 2205 cycle rows aligned by row position.")

All 17 sensor files have 2205 cycle rows aligned by row position.


## 8. Class distributions

In [11]:
distribution_rows = []
for target in config.TARGET_COLUMNS:
    counts = profile[target].value_counts().sort_index()
    ratio = counts.max() / counts.min()
    for value, count in counts.items():
        distribution_rows.append({"target": target, "value": int(value), "count": int(count),
                                  "percentage": round(100 * count / len(profile), 2),
                                  "largest_smallest_ratio": round(ratio, 3)})
    fig, ax = plt.subplots(figsize=(6, 4))
    counts.plot.bar(ax=ax)
    ax.set(title=f"{target} class distribution", xlabel="class", ylabel="cycles")
    save_figure(fig, f"01_class_distribution_{target}")
    plt.close(fig)

distributions = pd.DataFrame(distribution_rows)
distributions.to_csv(config.RESULTS_DIR / "class_distributions.csv", index=False)
display(distributions)

,target,value,count,percentage,largest_smallest_ratio
0,cooler,3,732,33.20,1.012
1,cooler,20,732,33.20,1.012
2,cooler,100,741,33.61,1.012
3,valve,73,360,16.33,3.125
4,valve,80,360,16.33,3.125
5,valve,90,360,16.33,3.125
6,valve,100,1125,51.02,3.125
7,pump,0,1221,55.37,2.482
8,pump,1,492,22.31,2.482
9,pump,2,492,22.31,2.482


### Initial class-distribution observation
Cooler classes are nearly balanced (ratio 1.012). Valve is most imbalanced (100 = 1125 cycles versus 360 for each other class; ratio 3.125). Pump is 1221/492/492 (ratio 2.482), accumulator is 808/599/399/399 (ratio 2.025), and stable is 1449/756 (ratio 1.917).

## 9. Label combinations and row order

In [12]:
combinations = check_label_combinations(profile)
combinations.to_csv(config.RESULTS_DIR / "label_combinations.csv", index=False)
print("Unique four-target combinations:", len(combinations))
print("Top combinations:")
display(combinations.head(10))
print("Bottom combinations:")
display(combinations.tail(10))

order = row_order_check(profile)
order.to_csv(config.RESULTS_DIR / "row_order_check.csv", index=False)
display(order)

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, target in zip(axes, config.TARGET_COLUMNS[:4]):
    ax.step(profile.index, profile[target], where="mid")
    ax.set_ylabel(target)
axes[-1].set_xlabel("row index")
save_figure(fig, "01_row_order_labels")
plt.close(fig)

Unique four-target combinations: 144
Top combinations:


,cooler,valve,pump,accumulator,count
0,100,100,0,90,230
1,3,100,0,130,221
2,20,100,0,90,221
3,3,100,0,90,21
4,3,100,0,100,21
5,3,100,0,115,21
6,20,100,0,100,21
7,20,100,0,115,21
8,20,100,0,130,21
9,100,100,0,100,21


Bottom combinations:


,cooler,valve,pump,accumulator,count
134,100,90,0,115,10
135,100,90,0,130,10
136,100,90,1,90,10
137,100,90,1,100,10
138,100,90,1,115,10
139,100,90,1,130,10
140,100,90,2,90,10
141,100,90,2,100,10
142,100,90,2,115,10
143,100,90,2,130,10


,target,transitions,observed_changes,observed_change_rate,expected_random_changes,expected_random_change_rate,ordering
0,cooler,2204,2,0.000907,1469.308855,0.666656,grouped
1,valve,2204,144,0.065336,1454.034152,0.659725,grouped
2,pump,2204,36,0.016334,1308.728652,0.593797,grouped
3,accumulator,2204,11,0.004991,1601.068574,0.726438,grouped


### Row-order implications
Cooler changed 2 times, valve 144, pump 36, and accumulator 11 across 2204 adjacent rows. These rates are far below the random-shuffle expectations (around 66.7%, 66.0%, 59.4%, and 72.6%). The conditions are grouped by row. A future train/test split must avoid accidental row slicing; use a reproducible split that checks condition coverage and follows the project Sprint 3 policy.

## 10. Report-only data-quality checks

In [13]:
quality = sensor_quality_checks()
quality.to_csv(config.RESULTS_DIR / "data_quality_checks.csv", index=False)
display(quality)

stable_counts = profile["stable"].value_counts().sort_index()
print("stable flag counts:", stable_counts.to_dict())

,sensor,NaN_count,inf_count,duplicate_full_cycles,constant_cycles,constant_columns,observed_min,observed_max
0,PS1,0,0,0,0,0,133.130005,191.919998
1,PS2,0,0,0,0,0,0.000000,167.770004
2,PS3,0,0,0,0,26,0.000000,18.827999
3,PS4,0,0,1237,1238,0,0.000000,10.266000
4,PS5,0,0,0,0,0,8.318000,10.041000
5,PS6,0,0,0,0,0,8.268000,9.910000
6,EPS1,0,0,0,0,0,2097.800049,2995.199951
7,FS1,0,0,0,0,0,0.000000,20.479000
8,FS2,0,0,0,0,0,8.764000,10.453000
9,TS1,0,0,0,0,0,34.984001,58.207001


stable flag counts: {0: 1449, 1: 756}


### Initial data-quality observations
- All 17 sensor files and `profile.txt` passed shape and missing-value validation; no NaN or infinite values were found.
- PS4 has 1237 duplicate complete cycles and 1238 constant cycles, while SE has 8 constant columns. These are report findings, not removal decisions.
- PS3 has 26 constant columns. No sensor has constant full cycles except PS4.
- Observed ranges include PS2/FS1/SE values of 0, and SE reaches 100.6%; these should be assessed in Sprint 2 rather than clipped here.
- The stable flag has 1449 rows with 0 and 756 with 1.